# LangChain Language Models on Amazon Bedrock: LLMs, Chat Models, and Embeddings

## 1. Theory & Real-World Context

In the LangChain framework, language models are categorized into three distinct abstractions to handle different data shapes and use cases. Understanding these distinctions is critical for building RAG (Retrieval-Augmented Generation) pipelines and autonomous agents.

* **LLMs (`BedrockLLM`):** The simplest abstraction. It takes a raw string as input and returns a raw string as output.
* *Real-World Use:* Best suited for legacy instruction models, single-shot batch processing tasks (like summarizing a 10-page log file), or generating code where conversational history is irrelevant.


* **Chat Models (`ChatBedrockConverse`):** The modern standard. Instead of raw text, these models accept a structured list of *Messages* (System, Human, AI/Assistant). They natively support conversational context, multimodality (passing images), and tool calling (Function Calling).
* *Real-World Use:* Chatbots, AI coding assistants, and interactive data analysis agents that need to remember what was said 3 turns ago.


* **Text Embedding Models (`BedrockEmbeddings`):** These models do not generate human-readable text. They take text as input and output a dense vector array (e.g., 1024 floating-point numbers). This vector represents the *semantic meaning* of the text.
* *Real-World Use:* The backbone of RAG. You convert all your enterprise documents into embeddings and store them in a Vector Database (like Amazon OpenSearch). When a user asks a question, you embed the question and perform a mathematical similarity search to find the right document.



---

## 2. Visual Architecture

```mermaid
flowchart TD
    subgraph Input [Application Input]
        S[Raw String Prompt]
        M[Message Array: Human/AI]
        D[Enterprise Document]
    end

    subgraph LangChainAWS [langchain-aws Abstractions]
        LLM[BedrockLLM]
        CHAT[ChatBedrockConverse]
        EMB[BedrockEmbeddings]
    end

    subgraph AWSBedrock [Amazon Bedrock Data Plane]
        T[Amazon Titan Text]
        C[Anthropic Claude / Nova]
        E[Amazon Titan Embeddings v2]
    end
    
    subgraph Output [System Output]
        R1[Raw Text String]
        R2[AI Message Object]
        R3[(Vector Database)]
    end

    S --> LLM --> T --> R1
    M --> CHAT --> C --> R2
    D --> EMB --> E --> R3

```

---

## 3. Console Hands-On

**Objective:** Provision a Custom Model ARN for a specialized fine-tuned workload (e.g., a model fine-tuned on your company's proprietary test engineering logs).

1. Navigate to the **Amazon Bedrock Console**.
2. In the left navigation pane, under **Foundation models**, click **Custom models**.
3. Click **Customize model** $\rightarrow$ **Fine-tune model**.
4. Select a base model (e.g., *Amazon Titan Text* or *Cohere Command*).
5. Provide an **S3 URI** pointing to your JSONL training data (e.g., your historical test cases and resolutions).
6. Once the job completes, you must purchase **Provisioned Throughput** for the model to actively host it.
7. After provisioning, AWS generates a custom **Model ARN** (e.g., `arn:aws:bedrock:us-east-1:123456789012:provisioned-model/abc123xyz`). You will use this ARN in LangChain instead of a standard `model_id`.

---

## 4. Boto3 / LangChain Implementation

Below is a production-ready Python script demonstrating how to instantiate and use all three LangChain model abstractions securely using `boto3`.

```python
import logging
import boto3
from botocore.exceptions import ClientError
from langchain_aws import BedrockLLM, ChatBedrockConverse, BedrockEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage

# Configure structured logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("langchain_models")

class BedrockModelManager:
    """Manages different LangChain model abstractions for Bedrock."""

    def __init__(self, region_name: str = "us-east-1"):
        # Explicit Boto3 client for secure credential management in production
        self.bedrock_runtime = boto3.client("bedrock-runtime", region_name=region_name)

    def generate_batch_text(self, prompt: str) -> str:
        """Uses BedrockLLM for a simple, single-shot text completion."""
        try:
            llm = BedrockLLM(
                client=self.bedrock_runtime,
                model_id="amazon.titan-text-premier-v1:0",
                model_kwargs={"temperature": 0.1, "maxTokenCount": 512}
            )
            logger.info("Invoking BedrockLLM...")
            return llm.invoke(prompt)
        except ClientError as e:
            logger.error(f"LLM Error: {e.response['Error']['Message']}")
            return ""

    def chat_with_context(self, system_prompt: str, user_message: str) -> str:
        """Uses ChatBedrockConverse for message-based conversational AI."""
        try:
            chat = ChatBedrockConverse(
                client=self.bedrock_runtime,
                model="anthropic.claude-3-5-sonnet-20241022-v2:0",
                temperature=0.3,
                max_tokens=1024
            )
            
            messages = [
                SystemMessage(content=system_prompt),
                HumanMessage(content=user_message)
            ]
            
            logger.info("Invoking ChatBedrockConverse...")
            response = chat.invoke(messages)
            return response.content
        except ClientError as e:
            logger.error(f"Chat Error: {e.response['Error']['Message']}")
            return ""

    def generate_embeddings(self, text: str) -> list[float]:
        """Uses BedrockEmbeddings to generate a dense semantic vector."""
        try:
            embeddings_client = BedrockEmbeddings(
                client=self.bedrock_runtime,
                model_id="amazon.titan-embed-text-v2:0"
            )
            logger.info("Generating embedding vector...")
            # embed_query is optimized for search queries, embed_documents is for database ingestion
            return embeddings_client.embed_query(text)
        except ClientError as e:
            logger.error(f"Embedding Error: {e.response['Error']['Message']}")
            return []

if __name__ == "__main__":
    manager = BedrockModelManager()
    
    # 1. Test Chat Model
    chat_response = manager.chat_with_context(
        system_prompt="You are a senior AWS Solutions Architect.",
        user_message="Explain why embeddings are better than keyword search in 2 sentences."
    )
    print(f"\n[Chat Response]:\n{chat_response}")

    # 2. Test Embeddings
    vector = manager.generate_embeddings("How to deploy a scalable data pipeline.")
    if vector:
        print(f"\n[Embedding Success]: Generated vector of length {len(vector)}")
        print(f"[Vector Sample]: {vector[:5]} ...")

```

---

## 5. Semantic Similarity Example (Replaced Theme)

*As requested, the food/puppy example has been replaced with an AI Engineering context to demonstrate how embedding vectors capture mathematical similarity based on meaning, structure, and language.*

**Baseline:** `The AI engineer deployed the model to production.`

| Score (Lower = More Similar) | Candidate Text | What Changed / Why the Score? |
| --- | --- | --- |
| **0.0412** | The machine learning engineer released the model into the production environment. | **Synonyms:** Exact same semantic meaning, using different vocabulary. |
| **0.0895** | The model was deployed to production by the AI engineer. | **Structure:** Passive voice, but retains the core semantic action. |
| **0.1844** | The AI engineer tested the model in the staging environment. | **Different Action/Target:** Similar entities (engineer, model), but the core intent changed (testing vs deployment). |
| **0.3514** | El ingeniero de IA desplegó el modelo en producción. | **Cross-Lingual:** Captures the exact meaning despite being written entirely in Spanish. |
| **0.7892** | A mechanical engineer designed the car engine. | **Context Shift:** Uses the word "engineer", but the semantic context is physical engineering, not software/AI. |
| **1.0562** | The weather in Bengaluru is sunny with a chance of rain. | **Completely Unrelated:** No overlap in vocabulary or semantic meaning. |

---

## 6. Practice Challenges

### Challenge 1: The Vector Distance Calculator

**Scenario:** You need to write a script to deduplicate test bug reports based on meaning, not exact wording.

* **Task:** Use `BedrockEmbeddings` to embed two sentences:
1. `"The database connection timed out during the high-load test."`
2. `"Under heavy traffic, the DB failed to respond in time."`
Write a Python function using `numpy` or `scipy.spatial.distance.cosine` to calculate the cosine similarity between the two resulting vectors to prove they mean the same thing.



### Challenge 2: Provisioned Custom Model Integration

**Scenario:** Your team fine-tuned a Meta Llama 3 model on proprietary company data and purchased provisioned throughput.

* **Task:** Write a LangChain `BedrockLLM` initialization block. Instead of a standard `model_id`, pass the AWS ARN of your provisioned model. Include the `credentials_profile_name` parameter to assume a specific AWS CLI profile (e.g., `mlops-admin`) that has IAM permissions to access the provisioned resource.

---

## 7. Industry Standards & Tips

* **Standardize on `ChatBedrockConverse`:** The original `ChatBedrock` wrapper is rapidly becoming legacy. Always use `ChatBedrockConverse` as it supports AWS's unified Converse API, which natively handles advanced features like Tool Calling (Functions) and multi-modal image inputs much more cleanly than the older abstraction.
* **Embedding Dimensions:** When using `amazon.titan-embed-text-v2:0`, you can specify the vector dimension size (256, 512, or 1024) in the `model_kwargs`. Lower dimensions save massive amounts of storage cost in your Vector DB and speed up search, but sacrifice a small amount of accuracy.
* **Embed Query vs. Embed Documents:** In LangChain, use `embeddings.embed_documents([list_of_strings])` when chunking and saving your knowledge base to a database. Use `embeddings.embed_query("string")` when taking the user's live chat question. The underlying model often applies different optimizations for storage vs. lookup.

---

## 8. Interview Q&A (Targeted for AI Developers)

### Q1: In LangChain, what is the architectural difference between an `LLM` and a `ChatModel`?

> **Answer:** An `LLM` is a stateless abstraction that takes a single text string as input and returns a string as output. A `ChatModel` uses a structured message abstraction (`SystemMessage`, `HumanMessage`, `AIMessage`). While both may use the same underlying foundation model, the `ChatModel` interface is designed to maintain conversational boundaries, handle role-prompting securely, and seamlessly integrate with LangChain's memory components.

### Q2: We are building an enterprise search tool. Why do we need an Embedding model? Why can't we just use traditional keyword search (Elasticsearch)?

> **Answer:** Traditional keyword search struggles with vocabulary mismatch (e.g., searching "deployment failure" might miss a document describing a "release crash"). Embedding models solve this by converting text into dense mathematical vectors based on semantic meaning. This allows a Vector Database to perform a nearest-neighbor search to find conceptually relevant documents, even if the user's query shares zero exact words with the source document.

### Q3: You instantiate a `BedrockLLM` via LangChain on an AWS Lambda function, but it fails with a `ValidationException` stating you lack permissions to a specific ARN. You aren't using a custom model. What went wrong?

> **Answer:** It's likely an IAM permissions issue on the Lambda's Execution Role. By default, LangChain uses the ambient AWS credentials. The role must have the `bedrock:InvokeModel` permission. Furthermore, I must ensure that third-party model access (like Anthropic or Meta) has been explicitly requested and granted within the AWS Bedrock Console for that specific AWS account and region, as it is denied by default.